In [23]:
import pandas as pd
import numpy as np
import seaborn as sns
import unicodedata
import re
import os

os.getcwd()

'/Users/hdnovak/Documents/ciencias_sociais/economia I/economia'

In [24]:
df_munprop = pd.read_excel(f'{os.getcwd()}/dados_brutos/tabela_completa_municipios_proporcoes.xlsx')
df_filiados = pd.read_csv(f'{os.getcwd()}/dados_brutos/filiado_mensal_1.csv',encoding='latin1',sep=';')

#Tabela 10280
df_tab10280_norm = pd.read_csv(f'{os.getcwd()}/dados_brutos/tabela10280/tabela10280_normalizada.csv')
df_tab10280_priv_pub = pd.read_csv(f'{os.getcwd()}/dados_brutos/tabela10280/tabela10280_municipios_publico_privado.csv')
df_tab10280_ocup_rac_cor = pd.read_csv(f'{os.getcwd()}/dados_brutos/tabela10280/tabela10283_ocupacao_cor_raca.csv')

In [25]:
print('---- Para visão geral ----')
print(f'O formato do df de filiados é: {df_filiados.shape}')
print(f'O formato do df de munícipios é: {df_munprop.shape}')

# Tabela 10280
print('---- Para tabela 10280 ----')
print(f'O formato do df da tabela 10280 normalizada é: {df_tab10280_norm.shape}')
print(f'O formato do df da tabela 10280 Priv|Pub é: {df_tab10280_priv_pub.shape}')
print(f'O formato do df da tabela 10280 Ocup|Raça|Cor é: {df_tab10280_ocup_rac_cor.shape}')

---- Para visão geral ----
O formato do df de filiados é: (127466, 5)
O formato do df de munícipios é: (5711, 9)
---- Para tabela 10280 ----
O formato do df da tabela 10280 normalizada é: (61270, 13)
O formato do df da tabela 10280 Priv|Pub é: (5570, 30)
O formato do df da tabela 10280 Ocup|Raça|Cor é: (11556, 13)


In [26]:
print('---- Para visão geral ----')
print(f'As colunas do df de filiados são: {df_filiados.columns}')
print(f'As colunas do df de munícipios são: {df_munprop.columns}')

# Tabela 10280
print('---- Para tabela 10280 ----')
print(f'As colunas do df da tabela 10280 normalizada são: {df_tab10280_norm.columns}')
print(f'As colunas do df da tabela 10280 Priv|Pub são: {df_tab10280_priv_pub.columns}')
print(f'As colunas do df da tabela 10280 Ocup|Raça|Cor são: {df_tab10280_ocup_rac_cor.columns}')

---- Para visão geral ----
As colunas do df de filiados são: Index(['Municipio', 'UF', 'Sigla partido', 'Quantidade de filiados',
       'Data de carga'],
      dtype='str')
As colunas do df de munícipios são: Index(['cidade_uf', 'total_filiados', 'proporcao_maior_partido',
       'proporcao_segundo_maior', 'Municípios.POPULAÇÃO ESTIMADA',
       'Planilha1.Total', 'Planilha1.Empregado no setor público',
       'Planilha1.Proporcao_pub_tot', 'fil_por_mil'],
      dtype='str')
---- Para tabela 10280 ----
As colunas do df da tabela 10280 normalizada são: Index(['cod_ibge_7', 'uf', 'regiao', 'municipio', 'municipio_norm',
       'populacao_estimada', 'porte_municipio', 'ano', 'sexo',
       'posicao_categoria_emprego', 'posicao_categoria_emprego_norm',
       'pessoas', 'rendimento_medio_reais'],
      dtype='str')
As colunas do df da tabela 10280 Priv|Pub são: Index(['cod_ibge_7', 'uf', 'regiao', 'municipio', 'municipio_norm',
       'populacao_estimada', 'porte_municipio', 'ano', 'pesso

In [27]:
# ============================================================
# 1. Funções gerais de normalização
# ============================================================

def normalizar_texto(valor):
    """
    Normaliza textos para chaves de join:
    - minúsculas
    - sem acento
    - sem pontuação
    - espaços padronizados
    """
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()
    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(c for c in valor if not unicodedata.combining(c))
    valor = re.sub(r"[^a-z0-9\s]", " ", valor)
    valor = re.sub(r"\s+", " ", valor).strip()

    return valor


def normalizar_nome_coluna(coluna):
    """
    Normaliza nomes de colunas para snake_case simples.
    """
    coluna = normalizar_texto(coluna)
    coluna = coluna.replace(" ", "_")
    coluna = re.sub(r"_+", "_", coluna)
    coluna = coluna.strip("_")

    return coluna


def normalizar_colunas(df):
    """
    Aplica normalização nos nomes das colunas.
    """
    df = df.copy()
    df.columns = [normalizar_nome_coluna(c) for c in df.columns]

    return df


def classificar_regiao_por_uf(uf):
    """
    Cria regionalização brasileira a partir da UF.
    """
    if pd.isna(uf):
        return np.nan

    uf = str(uf).upper().strip()

    mapa_regiao = {
        # Norte
        "AC": "Norte",
        "AP": "Norte",
        "AM": "Norte",
        "PA": "Norte",
        "RO": "Norte",
        "RR": "Norte",
        "TO": "Norte",

        # Nordeste
        "AL": "Nordeste",
        "BA": "Nordeste",
        "CE": "Nordeste",
        "MA": "Nordeste",
        "PB": "Nordeste",
        "PE": "Nordeste",
        "PI": "Nordeste",
        "RN": "Nordeste",
        "SE": "Nordeste",

        # Centro-Oeste
        "DF": "Centro-Oeste",
        "GO": "Centro-Oeste",
        "MT": "Centro-Oeste",
        "MS": "Centro-Oeste",

        # Sudeste
        "ES": "Sudeste",
        "MG": "Sudeste",
        "RJ": "Sudeste",
        "SP": "Sudeste",

        # Sul
        "PR": "Sul",
        "RS": "Sul",
        "SC": "Sul",
    }

    return mapa_regiao.get(uf, np.nan)


def classificar_porte_municipio(populacao):
    """
    Classifica porte populacional dos municípios.
    """
    if pd.isna(populacao):
        return np.nan

    if populacao <= 10_000:
        return "Muito pequeno: até 10 mil hab."
    elif populacao <= 50_000:
        return "Pequeno: 10 mil a 50 mil hab."
    elif populacao <= 100_000:
        return "Médio: 50 mil a 100 mil hab."
    else:
        return "Grande: acima de 100 mil hab."


def converter_numero_br(valor):
    """
    Converte números em formato brasileiro.
    Exemplo:
    '22.787' -> 22787
    '1.234,56' -> 1234.56
    """
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    if valor in ["-", "..", "...", "X", "x", ""]:
        return np.nan

    valor = re.sub(r"\(\d+\)", "", valor).strip()
    valor = valor.replace(".", "").replace(",", ".")

    try:
        return float(valor)
    except ValueError:
        return np.nan

In [28]:
# ============================================================
# 2. Normalização inicial dos dataframes
# ============================================================

df_filiados_norm = normalizar_colunas(df_filiados)
df_munprop_norm = normalizar_colunas(df_munprop)
df_tab10280_norm = normalizar_colunas(df_tab10280_norm)
df_tab10280_priv_pub_norm = normalizar_colunas(df_tab10280_priv_pub)
df_tab10280_ocup_rac_cor_norm = normalizar_colunas(df_tab10280_ocup_rac_cor)

print("Colunas df_filiados_norm:")
print(df_filiados_norm.columns)

print("\nColunas df_munprop_norm:")
print(df_munprop_norm.columns)

print("\nColunas df_tab10280_priv_pub_norm:")
print(df_tab10280_priv_pub_norm.columns)

Colunas df_filiados_norm:
Index(['municipio', 'uf', 'sigla_partido', 'quantidade_de_filiados',
       'data_de_carga'],
      dtype='str')

Colunas df_munprop_norm:
Index(['cidade_uf', 'total_filiados', 'proporcao_maior_partido',
       'proporcao_segundo_maior', 'municipios_populacao_estimada',
       'planilha1_total', 'planilha1_empregado_no_setor_publico',
       'planilha1_proporcao_pub_tot', 'fil_por_mil'],
      dtype='str')

Colunas df_tab10280_priv_pub_norm:
Index(['cod_ibge_7', 'uf', 'regiao', 'municipio', 'municipio_norm',
       'populacao_estimada', 'porte_municipio', 'ano', 'pessoas_total',
       'pessoas_setor_publico', 'pessoas_setor_privado',
       'prop_setor_publico_sobre_total', 'prop_setor_privado_sobre_total',
       'renda_media_total', 'renda_media_setor_publico',
       'renda_media_setor_privado', 'diferenca_renda_publico_privado_rs',
       'diferenca_renda_publico_privado_pct', 'pessoas_publico_estatutario',
       'pessoas_publico_nao_estatutario', 'pesso

In [29]:
# ============================================================
# 3. Filiados: normalização de município, UF e partido
# ============================================================

df_filiados_norm["uf"] = (
    df_filiados_norm["uf"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df_filiados_norm["municipio_nome_padronizado"] = (
    df_filiados_norm["municipio"]
    .apply(normalizar_texto)
)

df_filiados_norm["sigla_partido"] = (
    df_filiados_norm["sigla_partido"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df_filiados_norm["quantidade_filiados"] = pd.to_numeric(
    df_filiados_norm["quantidade_de_filiados"],
    errors="coerce"
)

df_filiados_norm["regiao"] = df_filiados_norm["uf"].apply(classificar_regiao_por_uf)

df_filiados_norm["chave_municipio_uf"] = (
    df_filiados_norm["municipio_nome_padronizado"]
    + "_"
    + df_filiados_norm["uf"]
)

In [30]:
# ============================================================
# 4. Munprop: normalização da base municipal já existente
# ============================================================

# Renomear colunas para nomes mais autoexplicativos
df_munprop_norm = df_munprop_norm.rename(columns={
    "municipios_populacao_estimada": "populacao_estimada",
    "planilha1_total": "total_trabalhadores",
    "planilha1_empregado_no_setor_publico": "trabalhadores_setor_publico",
    "planilha1_proporcao_pub_tot": "proporcao_trabalhadores_setor_publico",
    "fil_por_mil": "filiados_por_mil_habitantes"
})

# Extrair UF do campo cidade_uf
df_munprop_norm["uf"] = (
    df_munprop_norm["cidade_uf"]
    .astype(str)
    .str.extract(r"([A-Z]{2})$")[0]
)

# Extrair município do campo cidade_uf
df_munprop_norm["municipio_nome_original"] = (
    df_munprop_norm["cidade_uf"]
    .astype(str)
    .str.replace(r"[-_/]\s*[A-Z]{2}$", "", regex=True)
    .str.replace(r"\s+[A-Z]{2}$", "", regex=True)
    .str.strip()
)

df_munprop_norm["municipio_nome_padronizado"] = (
    df_munprop_norm["municipio_nome_original"]
    .apply(normalizar_texto)
)

df_munprop_norm["regiao"] = df_munprop_norm["uf"].apply(classificar_regiao_por_uf)

df_munprop_norm["chave_municipio_uf"] = (
    df_munprop_norm["municipio_nome_padronizado"]
    + "_"
    + df_munprop_norm["uf"]
)

# Conversões numéricas defensivas
cols_numericas_munprop = [
    "total_filiados",
    "proporcao_maior_partido",
    "proporcao_segundo_maior",
    "populacao_estimada",
    "total_trabalhadores",
    "trabalhadores_setor_publico",
    "proporcao_trabalhadores_setor_publico",
    "filiados_por_mil_habitantes"
]

for col in cols_numericas_munprop:
    if col in df_munprop_norm.columns:
        df_munprop_norm[col] = pd.to_numeric(df_munprop_norm[col], errors="coerce")

# Colunas derivadas
df_munprop_norm["porte_municipio"] = (
    df_munprop_norm["populacao_estimada"]
    .apply(classificar_porte_municipio)
)

df_munprop_norm["log_populacao_estimada"] = np.log1p(
    df_munprop_norm["populacao_estimada"]
)

df_munprop_norm["trabalhadores_publicos_por_mil_habitantes"] = (
    df_munprop_norm["trabalhadores_setor_publico"]
    / df_munprop_norm["populacao_estimada"]
    * 1000
)

In [31]:
# ============================================================
# 5. Tabela 10280 público/privado: nomes melhores e regionalização
# ============================================================

df_tab10280_priv_pub_norm = df_tab10280_priv_pub_norm.rename(columns={
    "cod_ibge_7": "codigo_ibge_municipio",
    "uf": "uf",
    "regiao": "regiao_original",
    "municipio": "municipio_nome_original",
    "municipio_norm": "municipio_nome_padronizado",
    "populacao_estimada": "populacao_estimada_ibge",
    "porte_municipio": "porte_municipio_original",
    "ano": "ano_referencia",
    "pessoas_total": "total_pessoas_ocupadas",
    "pessoas_setor_publico": "pessoas_ocupadas_setor_publico",
    "pessoas_setor_privado": "pessoas_ocupadas_setor_privado",
    "prop_setor_publico_sobre_total": "proporcao_ocupados_setor_publico",
    "prop_setor_privado_sobre_total": "proporcao_ocupados_setor_privado",
    "renda_media_total": "renda_media_total_reais",
    "renda_media_setor_publico": "renda_media_setor_publico_reais",
    "renda_media_setor_privado": "renda_media_setor_privado_reais",
    "diferenca_renda_publico_privado_rs": "diferenca_renda_publico_privado_reais",
    "diferenca_renda_publico_privado_pct": "diferenca_percentual_renda_publico_privado",
    "pessoas_publico_estatutario": "pessoas_ocupadas_publico_estatutario",
    "pessoas_publico_nao_estatutario": "pessoas_ocupadas_publico_nao_estatutario",
    "pessoas_empresas_estatais": "pessoas_ocupadas_empresas_estatais",
    "pessoas_trabalhador_domestico": "pessoas_ocupadas_trabalho_domestico",
    "pessoas_empregador": "pessoas_ocupadas_empregadores",
    "pessoas_conta_propria": "pessoas_ocupadas_conta_propria",
    "renda_media_publico_estatutario": "renda_media_publico_estatutario_reais",
    "renda_media_publico_nao_estatutario": "renda_media_publico_nao_estatutario_reais",
    "renda_media_empresas_estatais": "renda_media_empresas_estatais_reais",
    "renda_media_trabalhador_domestico": "renda_media_trabalho_domestico_reais",
    "renda_media_empregador": "renda_media_empregadores_reais",
    "renda_media_conta_propria": "renda_media_conta_propria_reais"
})

df_tab10280_priv_pub_norm["uf"] = (
    df_tab10280_priv_pub_norm["uf"]
    .astype(str)
    .str.upper()
    .str.strip()
)

df_tab10280_priv_pub_norm["municipio_nome_padronizado"] = (
    df_tab10280_priv_pub_norm["municipio_nome_padronizado"]
    .apply(normalizar_texto)
)

df_tab10280_priv_pub_norm["regiao"] = (
    df_tab10280_priv_pub_norm["uf"]
    .apply(classificar_regiao_por_uf)
)

df_tab10280_priv_pub_norm["chave_municipio_uf"] = (
    df_tab10280_priv_pub_norm["municipio_nome_padronizado"]
    + "_"
    + df_tab10280_priv_pub_norm["uf"]
)

df_tab10280_priv_pub_norm["porte_municipio"] = (
    df_tab10280_priv_pub_norm["populacao_estimada_ibge"]
    .apply(classificar_porte_municipio)
)

df_tab10280_priv_pub_norm["log_populacao_estimada"] = np.log1p(
    df_tab10280_priv_pub_norm["populacao_estimada_ibge"]
)

# Métricas econômicas adicionais
df_tab10280_priv_pub_norm["razao_renda_publico_privado"] = (
    df_tab10280_priv_pub_norm["renda_media_setor_publico_reais"]
    / df_tab10280_priv_pub_norm["renda_media_setor_privado_reais"]
)

df_tab10280_priv_pub_norm["premio_salarial_publico_percentual"] = (
    df_tab10280_priv_pub_norm["razao_renda_publico_privado"] - 1
)

df_tab10280_priv_pub_norm["ocupados_setor_publico_por_mil_habitantes"] = (
    df_tab10280_priv_pub_norm["pessoas_ocupadas_setor_publico"]
    / df_tab10280_priv_pub_norm["populacao_estimada_ibge"]
    * 1000
)

df_tab10280_priv_pub_norm["ocupados_setor_privado_por_mil_habitantes"] = (
    df_tab10280_priv_pub_norm["pessoas_ocupadas_setor_privado"]
    / df_tab10280_priv_pub_norm["populacao_estimada_ibge"]
    * 1000
)

In [32]:
# ============================================================
# 6. Agregar filiados por município
# ============================================================

df_filiados_municipio = (
    df_filiados_norm
    .groupby(
        [
            "chave_municipio_uf",
            "municipio_nome_padronizado",
            "uf",
            "regiao"
        ],
        as_index=False
    )
    .agg(
        total_filiados_tse=("quantidade_filiados", "sum"),
        quantidade_partidos_com_filiados=("sigla_partido", "nunique")
    )
)

# Ranking de partidos por município
df_filiados_partido = (
    df_filiados_norm
    .groupby(
        [
            "chave_municipio_uf",
            "municipio_nome_padronizado",
            "uf",
            "regiao",
            "sigla_partido"
        ],
        as_index=False
    )
    .agg(
        filiados_partido=("quantidade_filiados", "sum")
    )
)

df_filiados_partido["ranking_partido_no_municipio"] = (
    df_filiados_partido
    .groupby("chave_municipio_uf")["filiados_partido"]
    .rank(method="first", ascending=False)
)

df_top_partidos = (
    df_filiados_partido
    .query("ranking_partido_no_municipio <= 2")
    .pivot_table(
        index=[
            "chave_municipio_uf",
            "municipio_nome_padronizado",
            "uf",
            "regiao"
        ],
        columns="ranking_partido_no_municipio",
        values="filiados_partido",
        aggfunc="first"
    )
    .reset_index()
    .rename(columns={
        1.0: "filiados_maior_partido",
        2.0: "filiados_segundo_maior_partido"
    })
)

df_filiados_municipio = df_filiados_municipio.merge(
    df_top_partidos,
    on=[
        "chave_municipio_uf",
        "municipio_nome_padronizado",
        "uf",
        "regiao"
    ],
    how="left"
)

df_filiados_municipio["proporcao_filiados_maior_partido"] = (
    df_filiados_municipio["filiados_maior_partido"]
    / df_filiados_municipio["total_filiados_tse"]
)

df_filiados_municipio["proporcao_filiados_segundo_maior_partido"] = (
    df_filiados_municipio["filiados_segundo_maior_partido"]
    / df_filiados_municipio["total_filiados_tse"]
)

df_filiados_municipio["concentracao_filiados_dois_maiores_partidos"] = (
    df_filiados_municipio["proporcao_filiados_maior_partido"]
    + df_filiados_municipio["proporcao_filiados_segundo_maior_partido"]
)

In [33]:
# ============================================================
# 7. Base final municipal
# ============================================================

# ----------------------------
# 7.1 Selecionar colunas da base de filiados
# ----------------------------

cols_filiados_para_join = [
    "chave_municipio_uf",
    "total_filiados_tse",
    "quantidade_partidos_com_filiados",
    "filiados_maior_partido",
    "filiados_segundo_maior_partido",
    "proporcao_filiados_maior_partido",
    "proporcao_filiados_segundo_maior_partido",
    "concentracao_filiados_dois_maiores_partidos"
]

cols_filiados_para_join = [
    col for col in cols_filiados_para_join
    if col in df_filiados_municipio.columns
]

df_municipios_base = df_tab10280_priv_pub_norm.merge(
    df_filiados_municipio[cols_filiados_para_join],
    on="chave_municipio_uf",
    how="left"
)

# ----------------------------
# 7.2 Selecionar e renomear colunas da base munprop
# ----------------------------

cols_munprop_para_join = [
    "chave_municipio_uf",
    "total_filiados",
    "filiados_por_mil_habitantes",
    "proporcao_maior_partido",
    "proporcao_segundo_maior",
    "total_trabalhadores",
    "trabalhadores_setor_publico",
    "proporcao_trabalhadores_setor_publico"
]

cols_munprop_para_join = [
    col for col in cols_munprop_para_join
    if col in df_munprop_norm.columns
]

df_munprop_join = df_munprop_norm[cols_munprop_para_join].copy()

df_munprop_join = df_munprop_join.rename(columns={
    "total_filiados": "total_filiados_munprop",
    "filiados_por_mil_habitantes": "filiados_por_mil_habitantes_munprop",
    "proporcao_maior_partido": "proporcao_maior_partido_munprop",
    "proporcao_segundo_maior": "proporcao_segundo_maior_munprop",
    "total_trabalhadores": "total_trabalhadores_munprop",
    "trabalhadores_setor_publico": "trabalhadores_setor_publico_munprop",
    "proporcao_trabalhadores_setor_publico": "proporcao_trabalhadores_setor_publico_munprop"
})

df_municipios_base = df_municipios_base.merge(
    df_munprop_join,
    on="chave_municipio_uf",
    how="left"
)

# ----------------------------
# 7.3 Garantir existência das colunas necessárias
# ----------------------------

if "total_filiados_tse" not in df_municipios_base.columns:
    df_municipios_base["total_filiados_tse"] = np.nan

if "total_filiados_munprop" not in df_municipios_base.columns:
    df_municipios_base["total_filiados_munprop"] = np.nan

# ----------------------------
# 7.4 Definir total de filiados principal
# ----------------------------

df_municipios_base["total_filiados_final"] = (
    df_municipios_base["total_filiados_tse"]
    .combine_first(df_municipios_base["total_filiados_munprop"])
)

# Mantém também o nome mais simples
df_municipios_base["total_filiados"] = df_municipios_base["total_filiados_final"]

# ----------------------------
# 7.5 Métricas centrais da pesquisa
# ----------------------------

df_municipios_base["filiados_por_mil_habitantes"] = (
    df_municipios_base["total_filiados_final"]
    / df_municipios_base["populacao_estimada_ibge"]
    * 1000
)

df_municipios_base["filiados_por_pessoa_ocupada"] = (
    df_municipios_base["total_filiados_final"]
    / df_municipios_base["total_pessoas_ocupadas"]
)

df_municipios_base["filiados_por_ocupado_setor_publico"] = (
    df_municipios_base["total_filiados_final"]
    / df_municipios_base["pessoas_ocupadas_setor_publico"]
)

df_municipios_base["cidade_pequena_ate_50_mil"] = (
    df_municipios_base["populacao_estimada_ibge"] <= 50_000
)

df_municipios_base["cidade_muito_pequena_ate_10_mil"] = (
    df_municipios_base["populacao_estimada_ibge"] <= 10_000
)

# ----------------------------
# 7.6 Checagens rápidas
# ----------------------------

print("Shape df_municipios_base:", df_municipios_base.shape)

print("\nColunas relacionadas a filiados:")
print([col for col in df_municipios_base.columns if "filiado" in col])

print("\nMissing total_filiados_tse:")
print(df_municipios_base["total_filiados_tse"].isna().mean())

print("\nMissing total_filiados_munprop:")
print(df_municipios_base["total_filiados_munprop"].isna().mean())

print("\nMissing total_filiados_final:")
print(df_municipios_base["total_filiados_final"].isna().mean())

Shape df_municipios_base: (5570, 59)

Colunas relacionadas a filiados:
['total_filiados_tse', 'quantidade_partidos_com_filiados', 'filiados_maior_partido', 'filiados_segundo_maior_partido', 'proporcao_filiados_maior_partido', 'proporcao_filiados_segundo_maior_partido', 'concentracao_filiados_dois_maiores_partidos', 'total_filiados_munprop', 'filiados_por_mil_habitantes_munprop', 'total_filiados_final', 'total_filiados', 'filiados_por_mil_habitantes', 'filiados_por_pessoa_ocupada', 'filiados_por_ocupado_setor_publico']

Missing total_filiados_tse:
0.0025134649910233393

Missing total_filiados_munprop:
0.0025134649910233393

Missing total_filiados_final:
0.0025134649910233393


In [34]:
# ============================================================
# 8. Z-scores regionais
# ============================================================

cols_para_zscore_regional = [
    "filiados_por_mil_habitantes",
    "proporcao_ocupados_setor_publico",
    "premio_salarial_publico_percentual",
    "diferenca_renda_publico_privado_reais",
    "ocupados_setor_publico_por_mil_habitantes",
    "concentracao_filiados_dois_maiores_partidos"
]

for col in cols_para_zscore_regional:
    if col in df_municipios_base.columns:
        df_municipios_base[f"zscore_regional_{col}"] = (
            df_municipios_base
            .groupby("regiao")[col]
            .transform(lambda x: (x - x.mean()) / x.std())
        )

# Score exploratório da hipótese
df_municipios_base["score_exploratorio_hipotese"] = (
    df_municipios_base["zscore_regional_filiados_por_mil_habitantes"]
    + df_municipios_base["zscore_regional_proporcao_ocupados_setor_publico"]
    + df_municipios_base["zscore_regional_premio_salarial_publico_percentual"]
)

# Classificação qualitativa dos municípios
df_municipios_base["perfil_municipio_hipotese"] = np.select(
    [
        (
            df_municipios_base["zscore_regional_filiados_por_mil_habitantes"] > 1
        )
        & (
            df_municipios_base["zscore_regional_proporcao_ocupados_setor_publico"] > 1
        ),

        (
            df_municipios_base["zscore_regional_filiados_por_mil_habitantes"] > 1
        )
        & (
            df_municipios_base["zscore_regional_proporcao_ocupados_setor_publico"] <= 0
        ),

        (
            df_municipios_base["zscore_regional_filiados_por_mil_habitantes"] <= 0
        )
        & (
            df_municipios_base["zscore_regional_proporcao_ocupados_setor_publico"] > 1
        ),
    ],
    [
        "Alta filiação e alto peso do setor público",
        "Alta filiação e baixo peso do setor público",
        "Baixa filiação e alto peso do setor público",
    ],
    default="Perfil intermediário"
)

In [35]:
# ============================================================
# 9. Seleção final de colunas para análise
# ============================================================

colunas_finais = [
    # Identificação
    "codigo_ibge_municipio",
    "uf",
    "regiao",
    "municipio_nome_original",
    "municipio_nome_padronizado",
    "chave_municipio_uf",
    "ano_referencia",

    # Porte
    "populacao_estimada_ibge",
    "log_populacao_estimada",
    "porte_municipio",
    "cidade_pequena_ate_50_mil",
    "cidade_muito_pequena_ate_10_mil",

    # Filiação partidária
    "total_filiados",
    "filiados_por_mil_habitantes",
    "quantidade_partidos_com_filiados",
    "filiados_maior_partido",
    "filiados_segundo_maior_partido",
    "proporcao_filiados_maior_partido",
    "proporcao_filiados_segundo_maior_partido",
    "concentracao_filiados_dois_maiores_partidos",

    # Mercado de trabalho
    "total_pessoas_ocupadas",
    "pessoas_ocupadas_setor_publico",
    "pessoas_ocupadas_setor_privado",
    "proporcao_ocupados_setor_publico",
    "proporcao_ocupados_setor_privado",
    "ocupados_setor_publico_por_mil_habitantes",
    "ocupados_setor_privado_por_mil_habitantes",

    # Renda
    "renda_media_total_reais",
    "renda_media_setor_publico_reais",
    "renda_media_setor_privado_reais",
    "diferenca_renda_publico_privado_reais",
    "diferenca_percentual_renda_publico_privado",
    "razao_renda_publico_privado",
    "premio_salarial_publico_percentual",

    # Categorias específicas de ocupação
    "pessoas_ocupadas_publico_estatutario",
    "pessoas_ocupadas_publico_nao_estatutario",
    "pessoas_ocupadas_empresas_estatais",
    "pessoas_ocupadas_empregadores",
    "pessoas_ocupadas_conta_propria",
    "renda_media_publico_estatutario_reais",
    "renda_media_publico_nao_estatutario_reais",
    "renda_media_empresas_estatais_reais",
    "renda_media_empregadores_reais",
    "renda_media_conta_propria_reais",

    # Indicadores derivados
    "filiados_por_pessoa_ocupada",
    "filiados_por_ocupado_setor_publico",

    # Z-scores e perfil
    "zscore_regional_filiados_por_mil_habitantes",
    "zscore_regional_proporcao_ocupados_setor_publico",
    "zscore_regional_premio_salarial_publico_percentual",
    "zscore_regional_diferenca_renda_publico_privado_reais",
    "zscore_regional_ocupados_setor_publico_por_mil_habitantes",
    "zscore_regional_concentracao_filiados_dois_maiores_partidos",
    "score_exploratorio_hipotese",
    "perfil_municipio_hipotese"
]

colunas_finais_existentes = [
    col for col in colunas_finais
    if col in df_municipios_base.columns
]

df_analise_municipios = df_municipios_base[colunas_finais_existentes].copy()

print("Shape df_municipios_base:", df_municipios_base.shape)
print("Shape df_analise_municipios:", df_analise_municipios.shape)

df_analise_municipios.head()

Shape df_municipios_base: (5570, 67)
Shape df_analise_municipios: (5570, 54)


,codigo_ibge_municipio,uf,regiao,municipio_nome_original,municipio_nome_padronizado,chave_municipio_uf,ano_referencia,populacao_estimada_ibge,log_populacao_estimada,porte_municipio,...,filiados_por_pessoa_ocupada,filiados_por_ocupado_setor_publico,zscore_regional_filiados_por_mil_habitantes,zscore_regional_proporcao_ocupados_setor_publico,zscore_regional_premio_salarial_publico_percentual,zscore_regional_diferenca_renda_publico_privado_reais,zscore_regional_ocupados_setor_publico_por_mil_habitantes,zscore_regional_concentracao_filiados_dois_maiores_partidos,score_exploratorio_hipotese,perfil_municipio_hipotese
0,1100015,RO,Norte,Alta Floresta D'Oeste,alta floresta d oeste,alta floresta d oeste_RO,2022,22787,10.033989,Pequeno: 10 mil a 50 mil hab.,...,0.235327,1.265147,-0.620451,-0.623711,0.879973,1.289060,-0.058050,-1.215751,-0.364189,Perfil intermediário
1,1100023,RO,Norte,Ariquemes,ariquemes,ariquemes_RO,2022,109170,11.600671,Grande: acima de 100 mil hab.,...,0.155142,1.195105,-0.991637,-1.222030,0.702427,1.905282,-0.739333,-1.055101,-1.511240,Perfil intermediário
2,1100031,RO,Norte,Cabixi,cabixi,cabixi_RO,2022,5664,8.642062,Muito pequeno: até 10 mil hab.,...,0.440945,2.036364,0.524494,-0.298663,-0.199062,0.324259,0.455667,0.697379,0.026769,Perfil intermediário
3,1100049,RO,Norte,Cacoal,cacoal,cacoal_RO,2022,98280,11.495586,Médio: 50 mil a 100 mil hab.,...,0.176877,1.168870,-0.896251,-0.993021,1.527081,3.114703,-0.475733,-0.767415,-0.362191,Perfil intermediário
4,1100056,RO,Norte,Cerejeiras,cerejeiras,cerejeiras_RO,2022,16966,9.739026,Pequeno: 10 mil a 50 mil hab.,...,0.284161,1.714045,-0.250884,-0.839047,0.613090,1.609527,-0.168975,1.275506,-0.476842,Perfil intermediário


In [36]:
# ============================================================
# 10. Visão regional agregada
# ============================================================

df_analise_regional = (
    df_analise_municipios
    .groupby("regiao", as_index=False)
    .agg(
        quantidade_municipios=("codigo_ibge_municipio", "nunique"),
        populacao_total=("populacao_estimada_ibge", "sum"),
        total_filiados=("total_filiados", "sum"),
        media_filiados_por_mil_habitantes=("filiados_por_mil_habitantes", "mean"),
        mediana_filiados_por_mil_habitantes=("filiados_por_mil_habitantes", "median"),
        media_proporcao_ocupados_setor_publico=("proporcao_ocupados_setor_publico", "mean"),
        mediana_proporcao_ocupados_setor_publico=("proporcao_ocupados_setor_publico", "median"),
        media_renda_setor_publico=("renda_media_setor_publico_reais", "mean"),
        media_renda_setor_privado=("renda_media_setor_privado_reais", "mean"),
        media_diferenca_renda_publico_privado=("diferenca_renda_publico_privado_reais", "mean"),
        media_premio_salarial_publico=("premio_salarial_publico_percentual", "mean")
    )
)

df_analise_regional["filiados_por_mil_habitantes_agregado"] = (
    df_analise_regional["total_filiados"]
    / df_analise_regional["populacao_total"]
    * 1000
)

df_analise_regional

,regiao,quantidade_municipios,populacao_total,total_filiados,media_filiados_por_mil_habitantes,mediana_filiados_por_mil_habitantes,media_proporcao_ocupados_setor_publico,mediana_proporcao_ocupados_setor_publico,media_renda_setor_publico,media_renda_setor_privado,media_diferenca_renda_publico_privado,media_premio_salarial_publico,filiados_por_mil_habitantes_agregado
0,Centro-Oeste,467,17232941,1522229.0,183.467184,158.371041,0.172040,0.159074,3125.425675,1992.396767,1133.028908,0.573935,88.332514
1,Nordeste,1794,57244485,3970196.0,111.343551,97.752274,0.235535,0.221619,2173.273250,1128.254281,1045.018969,0.966869,69.355083
2,Norte,450,18801282,1504515.0,138.838647,118.470279,0.244585,0.226564,2477.056956,1453.759244,1023.297711,0.745860,80.021937
3,Sudeste,1668,88825643,5922241.0,133.888993,117.480697,0.165845,0.148729,2695.406337,1801.763765,893.642572,0.521183,66.672650
4,Sul,1191,31310809,3133966.0,181.264610,161.266591,0.133941,0.125061,3276.046692,2066.209681,1209.837011,0.596001,100.092144


In [37]:
# ============================================================
# 11. Visão por região e porte municipal
# ============================================================

df_analise_regiao_porte = (
    df_analise_municipios
    .groupby(["regiao", "porte_municipio"], as_index=False)
    .agg(
        quantidade_municipios=("codigo_ibge_municipio", "nunique"),
        populacao_total=("populacao_estimada_ibge", "sum"),
        total_filiados=("total_filiados", "sum"),
        media_filiados_por_mil_habitantes=("filiados_por_mil_habitantes", "mean"),
        media_proporcao_ocupados_setor_publico=("proporcao_ocupados_setor_publico", "mean"),
        media_ocupados_setor_publico_por_mil=("ocupados_setor_publico_por_mil_habitantes", "mean"),
        media_renda_setor_publico=("renda_media_setor_publico_reais", "mean"),
        media_renda_setor_privado=("renda_media_setor_privado_reais", "mean"),
        media_premio_salarial_publico=("premio_salarial_publico_percentual", "mean"),
        media_concentracao_dois_maiores_partidos=("concentracao_filiados_dois_maiores_partidos", "mean")
    )
)

df_analise_regiao_porte["filiados_por_mil_habitantes_agregado"] = (
    df_analise_regiao_porte["total_filiados"]
    / df_analise_regiao_porte["populacao_total"]
    * 1000
)

df_analise_regiao_porte

,regiao,porte_municipio,quantidade_municipios,populacao_total,total_filiados,media_filiados_por_mil_habitantes,media_proporcao_ocupados_setor_publico,media_ocupados_setor_publico_por_mil,media_renda_setor_publico,media_renda_setor_privado,media_premio_salarial_publico,media_concentracao_dois_maiores_partidos,filiados_por_mil_habitantes_agregado
0,Centro-Oeste,Grande: acima de 100 mil hab.,27,10737210,703092.0,63.606870,0.105219,47.616138,4583.098148,2298.964074,0.972270,0.261972,65.481815
1,Centro-Oeste,Muito pequeno: até 10 mil hab.,242,1212093,266355.0,249.419209,0.208864,88.041420,2733.875331,1885.066901,0.464037,0.345217,219.747990
2,Centro-Oeste,Médio: 50 mil a 100 mil hab.,20,1426841,111508.0,80.956622,0.112627,49.978446,4023.171500,2274.061500,0.786595,0.280940,78.150263
3,Centro-Oeste,Pequeno: 10 mil a 50 mil hab.,178,3856797,441274.0,122.819740,0.138786,59.589386,3335.780225,2060.167640,0.639030,0.289981,114.414630
4,Nordeste,Grande: acima de 100 mil hab.,69,24617427,1215210.0,52.801222,0.126056,48.586215,3818.167101,1725.373478,1.192415,0.252865,49.363810
5,Nordeste,Muito pequeno: até 10 mil hab.,604,3512490,528406.0,164.696504,0.296188,75.884133,1940.557599,1060.982003,0.889700,0.317066,150.436300
6,Nordeste,Médio: 50 mil a 100 mil hab.,116,7964830,459361.0,58.506332,0.156383,51.366543,2750.328276,1355.326121,1.050590,0.247617,57.673673
7,Nordeste,Pequeno: 10 mil a 50 mil hab.,1005,21149738,1767219.0,89.428435,0.215734,60.677124,2133.595831,1101.479055,0.988099,0.272144,83.557489
8,Norte,Grande: acima de 100 mil hab.,29,9787530,630966.0,69.406724,0.165262,58.742373,3831.915172,1821.833103,1.107993,0.270616,64.466316
9,Norte,Muito pequeno: até 10 mil hab.,157,799493,155212.0,216.933014,0.292321,97.128404,2221.812420,1500.662102,0.517862,0.294534,194.138035


In [38]:
# ============================================================
# 12. Exportação segura para Sheets preservando códigos
# ============================================================

def exportar_csv_sheets(df, caminho, colunas_texto=None):
    df_export = df.copy()

    if colunas_texto is None:
        colunas_texto = []

    for col in colunas_texto:
        if col in df_export.columns:
            df_export[col] = df_export[col].astype("string")

    df_export.to_csv(
        caminho,
        index=False,
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        float_format="%.6f"
    )


colunas_texto_municipio = [
    "codigo_ibge_municipio",
    "uf",
    "chave_municipio_uf"
]

exportar_csv_sheets(
    df_municipios_base,
    "df_municipios_base_completa_normalizada.csv",
    colunas_texto=colunas_texto_municipio
)

exportar_csv_sheets(
    df_analise_municipios,
    "df_analise_municipios.csv",
    colunas_texto=colunas_texto_municipio
)

exportar_csv_sheets(
    df_analise_regional,
    "df_analise_regional.csv",
    colunas_texto=["regiao"]
)

exportar_csv_sheets(
    df_analise_regiao_porte,
    "df_analise_regiao_porte.csv",
    colunas_texto=["regiao", "porte_municipio"]
)

In [39]:
# ============================================================
# 13. Checagens de qualidade
# ============================================================

print("Municípios únicos:", df_analise_municipios["codigo_ibge_municipio"].nunique())
print("Linhas:", len(df_analise_municipios))

print("\nMissing principais:")
display(
    df_analise_municipios[
        [
            "total_filiados",
            "filiados_por_mil_habitantes",
            "proporcao_ocupados_setor_publico",
            "renda_media_setor_publico_reais",
            "renda_media_setor_privado_reais",
            "premio_salarial_publico_percentual"
        ]
    ]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nDistribuição por região:")
display(
    df_analise_municipios["regiao"]
    .value_counts(dropna=False)
)

print("\nDistribuição por porte:")
display(
    df_analise_municipios["porte_municipio"]
    .value_counts(dropna=False)
)

print("\nTop 20 municípios mais aderentes à hipótese:")
display(
    df_analise_municipios
    .sort_values("score_exploratorio_hipotese", ascending=False)
    [
        [
            "municipio_nome_original",
            "uf",
            "regiao",
            "porte_municipio",
            "filiados_por_mil_habitantes",
            "proporcao_ocupados_setor_publico",
            "premio_salarial_publico_percentual",
            "score_exploratorio_hipotese",
            "perfil_municipio_hipotese"
        ]
    ]
    .head(20)
)

Municípios únicos: 5570
Linhas: 5570

Missing principais:


total_filiados                        0.002513
filiados_por_mil_habitantes           0.002513
proporcao_ocupados_setor_publico      0.000000
renda_media_setor_publico_reais       0.000000
renda_media_setor_privado_reais       0.000000
premio_salarial_publico_percentual    0.000000
dtype: float64


Distribuição por região:


regiao
Nordeste        1794
Sudeste         1668
Sul             1191
Centro-Oeste     467
Norte            450
Name: count, dtype: int64


Distribuição por porte:


porte_municipio
Muito pequeno: até 10 mil hab.    2468
Pequeno: 10 mil a 50 mil hab.     2420
Médio: 50 mil a 100 mil hab.       344
Grande: acima de 100 mil hab.      338
Name: count, dtype: int64


Top 20 municípios mais aderentes à hipótese:


,municipio_nome_original,uf,regiao,porte_municipio,filiados_por_mil_habitantes,proporcao_ocupados_setor_publico,premio_salarial_publico_percentual,score_exploratorio_hipotese,perfil_municipio_hipotese
2560,Grupiara,MG,Sudeste,Muito pequeno: até 10 mil hab.,507.002801,0.554149,0.372046,10.123280,Alta filiação e alto peso do setor público
1296,Carrapateira,PB,Nordeste,Muito pequeno: até 10 mil hab.,385.623679,0.551724,1.182966,9.212302,Alta filiação e alto peso do setor público
2753,Nacip Raydan,MG,Sudeste,Muito pequeno: até 10 mil hab.,433.333333,0.438168,0.771313,8.844903,Alta filiação e alto peso do setor público
1433,São José do Brejo do Cruz,PB,Nordeste,Muito pequeno: até 10 mil hab.,438.857143,0.599010,0.358306,8.750452,Alta filiação e alto peso do setor público
1320,Emas,PB,Nordeste,Muito pequeno: até 10 mil hab.,470.355731,0.421405,0.884504,8.489526,Alta filiação e alto peso do setor público
1415,Joca Claudino,PB,Nordeste,Muito pequeno: até 10 mil hab.,302.646720,0.650924,0.977386,8.387683,Alta filiação e alto peso do setor público
3004,São Sebastião do Rio Preto,MG,Sudeste,Muito pequeno: até 10 mil hab.,497.965826,0.438669,0.237566,7.957896,Alta filiação e alto peso do setor público
5550,Sítio d'Abadia,GO,Centro-Oeste,Muito pequeno: até 10 mil hab.,343.854936,0.591102,0.608609,7.940370,Alta filiação e alto peso do setor público
867,São Luis do Piauí,PI,Nordeste,Muito pequeno: até 10 mil hab.,352.217830,0.594118,0.685689,7.912660,Alta filiação e alto peso do setor público
3026,Serra da Saudade,MG,Sudeste,Muito pequeno: até 10 mil hab.,459.112150,0.530928,-0.091149,7.547198,Alta filiação e alto peso do setor público
